# Geocoding

The following section breaks down the process of geocoding violation location address and respondent home address in the following steps:

**1. Building and Cleaning Addresses:** This section uses various fields from `violations_df_full` to build violation and respondent addresses appropriately, then clean them before geocoding

**2. Saving Intersections for Separate Geocoding:** This section creates dataframes `violation_intersections.csv` and `respondent_intersections.csv` for addresses that are intersections. These will require separate geocoding

**3. Running Addresses through NYC GeoSupport Geocoder:** This section uses async batching to send multiple requests, reducing runtime.

**4. Flagging Levels of Geocoding Precision:**
This section adds the following tier categorizations for the geocoded addresses:

`0` if no match
`1` for an exact match
`2` for a fallback match on the same street
`3` for a fallback match in the same neighborhood
`4` for a fallback match with the same ZIP code
`5` for a fallback match in the same borough
`nan` if the matched address is something else

In [1]:
DATA_DIR = "/Users/amelie/Desktop/GitHub/nyc-street-vendor-enforcement/data/processed"

In [2]:
# Setup

import pandas as pd
import sys, os
sys.path.append(os.path.abspath("..")) 
from src.addresses import *
from src.geocoding import *

#DATA_DIR = "...data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

#Load df and modify datetime
violations_df_full = pd.read_csv(f"{DATA_DIR}/violations_df_full.csv")
violations_df_full['violation_date'] = pd.to_datetime(violations_df_full['violation_date'], errors='coerce')
violations_df_full['violation_year'] = violations_df_full['violation_date'].dt.year

/var/folders/dz/r3jh38h95rs7sxsc47pyxs980000gn/T/ipykernel_80317/2784160152.py:13: DtypeWarning: Columns (0: ticket_number, 1: respondent_address_zip_code, 2: charge_2_code, 3: charge_2_code_section, 4: charge_2_code_description) have mixed types. Specify dtype option on import or set low_memory=False.
  violations_df_full = pd.read_csv(f"{DATA_DIR}/violations_df_full.csv")


**Building and Cleaning Full Address Strings**

In [3]:
# Building full address
violations_df_full['violation_address'] = violations_df_full.apply(build_violation_address, axis=1)
violations_df_full['respondent_address'] = violations_df_full.apply(build_respondent_address, axis=1)

# Cleaning full address
violations_df_full['violation_address'] = violations_df_full['violation_address'].apply(clean_address)
violations_df_full['respondent_address'] = violations_df_full['respondent_address'].apply(clean_address)

**Geocoding**

In [21]:
# Deduplicating addresses to geocode
unique_violation_addresses = violations_df_full['violation_address'].drop_duplicates().to_frame()
unique_respondent_addresses = violations_df_full['respondent_address'].drop_duplicates().to_frame()

print("Number of unique violation addresses:", len(unique_violation_addresses))
print("\nNumber of unique respondent addresses:", len(unique_respondent_addresses), "\n")

# Geocoding
violation_results  = await geocode_addresses(
    unique_violation_addresses["violation_address"].tolist(),
    VIOLATION_CHECKPOINT
)

respondent_results = await geocode_addresses(
    unique_respondent_addresses["respondent_address"].tolist(),
    RESPONDENT_CHECKPOINT
)

print("\n\nGeocoding complete.")

Number of unique violation addresses: 38701

Number of unique respondent addresses: 14532 

Total:       38701
Already done:0
Remaining:   38701
  [17:09:59] Checkpointed 500/38701
  [17:11:53] Checkpointed 1000/38701
  [17:12:59] Checkpointed 1500/38701
  [17:14:34] Checkpointed 2000/38701
  [17:15:58] Checkpointed 2500/38701
  [17:17:51] Checkpointed 3000/38701
  [17:19:49] Checkpointed 3500/38701
  [17:21:29] Checkpointed 4000/38701
  [17:23:57] Checkpointed 4500/38701
  [17:25:53] Checkpointed 5000/38701
  [17:26:59] Checkpointed 5500/38701
  [17:28:32] Checkpointed 6000/38701
  [17:30:11] Checkpointed 6500/38701
  [17:31:58] Checkpointed 7000/38701
  [17:33:35] Checkpointed 7500/38701
  [17:35:42] Checkpointed 8000/38701
  [17:37:00] Checkpointed 8500/38701
  [17:38:59] Checkpointed 9000/38701
  [17:40:56] Checkpointed 9500/38701
  [17:43:01] Checkpointed 10000/38701
  [17:44:40] Checkpointed 10500/38701
  [17:45:52] Checkpointed 11000/38701
  [17:47:37] Checkpointed 11500/38701
 